In [3]:
import cv2
import numpy as np
from scipy.signal import convolve2d

In [ ]:
"""
HLAC特徴量 45次元対応表
------------------------

本HLAC特徴量ベクトルは、以下の45種類のマスク（局所自己相関パターン）に基づいています。
各特徴量は、画像上の各マスクパターンが出現する頻度をスケール不変＋正規化して表現したものです。

【1次自己相関（1st-order masks）: 3パターン】
    0: 中心画素
    1: 中心＋右隣
    2: 中心＋下隣

【2次自己相関（2nd-order masks）: 42パターン】
    以下は「中心+α+β」の2画素自己相関を意味します。
    (方向: 上=U, 下=D, 左=L, 右=R, 斜めUL, UR, DL, DR)

    3 : 中心+右+下      (R+D)
    4 : 中心+右+上      (R+U)
    5 : 中心+右+右      (R+R)
    6 : 中心+下+下      (D+D)
    7 : 中心+上+上      (U+U)
    8 : 中心+左+左      (L+L)
    9 : 中心+右+左      (R+L)
    10: 中心+下+上      (D+U)
    11: 中心+斜めUR+斜めDL
    12: 中心+斜めUL+斜めDR
    13: 中心+右+斜めDR
    14: 中心+右+斜めUR
    15: 中心+下+斜めDL
    16: 中心+下+斜めDR
    17: 中心+右+斜めDL
    18: 中心+下+斜めUR
    19: 中心+斜めUL+斜めUR
    20: 中心+斜めDL+斜めDR
    21: 中心+右+斜めUL
    22: 中心+下+斜めUL
    23: 中心+斜めUR+斜めDR
    24: 中心+斜めUL+斜めDL
    25: 中心+上+斜めUL
    26: 中心+左+斜めUL
    27: 中心+左+斜めDL
    28: 中心+上+斜めUR
    29: 中心+右+斜めDR
    30: 中心+上+斜めDR
    31: 中心+下+斜めDL
    32: 中心+上+斜めUL
    33: 中心+上+斜めUR
    34: 中心+左+斜めUL
    35: 中心+左+斜めDL
    36: 中心+右+斜めUL
    37: 中心+右+斜めUR
    38: 中心+下+斜めDL
    39: 中心+下+斜めDR
    40: 中心+上+右
    41: 中心+左+下
    42: 中心+右+下
    43: 中心+左+上
    44: 中心+左+右

備考:
- パターンは原論文「High-order local autocorrelation features for classifying textures and object shapes」(Otsu & Kurita, 1988) に基づく。
- 回転不変バージョンを使用する場合、これらのパターンは方向を変えても同一視される。
- 本リストはスケール不変の前提で設計され、入力サイズによらず特徴次元は45。

"""


In [ ]:
"""
HLAC回転不変特徴量（11次元）の対応パターン一覧

特徴量インデックス : 対応するピクセルパターン（中心を基準に隣接ピクセル）

0 : 中心のみ                               (C)
1 : 中心 + 右                            (C + R)
2 : 中心 + 右 + 右                     (C + R + R)
3 : 中心 + 右 + 下                     (C + R + D)
4 : 中心 + 右 + 上                     (C + R + U)
5 : 中心 + 下 + 下                     (C + D + D)
6 : 中心 + 右 + 右 + 下               (C + R + R + D)
7 : 中心 + 右 + 右 + 上               (C + R + R + U)
8 : 中心 + 右 + 下 + 下               (C + R + D + D)
9 : 中心 + 右 + 上 + 上               (C + R + U + U)
10: 中心 + 右 + 右 + 下 + 上          (C + R + R + D + U)

記号説明：
 C : 中心画素
 R : 右隣接画素
 L : 左隣接画素
 U : 上隣接画素
 D : 下隣接画素

備考：
- 回転不変なので、例えば (C + R + D) は (C + D + L) や (C + U + R) と同一視されます。
- パターンは論文の定義に合わせていますが、実装により微妙に異なることがあります。

"""


In [5]:
def hlac_kernels(order=2, rotate_invariant=False):
    """HLACのパターン生成（最大2次）"""
    patterns = []

    # 0次: 中心画素
    patterns.append(np.array([[0, 0, 0],
                              [0, 1, 0],
                              [0, 0, 0]], dtype=np.uint8))

    # 1次: 8方向
    offsets = [(0, -1), (-1, -1), (-1, 0), (-1, 1),
               (0, 1), (1, 1), (1, 0), (1, -1)]

    for dx, dy in offsets:
        k = np.zeros((3, 3), dtype=np.uint8)
        k[1, 1] = 1
        k[1 + dy, 1 + dx] = 1
        patterns.append(k)

    if order >= 2:
        for i in range(len(offsets)):
            for j in range(i, len(offsets)):
                dx1, dy1 = offsets[i]
                dx2, dy2 = offsets[j]
                k = np.zeros((3, 3), dtype=np.uint8)
                k[1, 1] = 1
                k[1 + dy1, 1 + dx1] = 1
                k[1 + dy2, 1 + dx2] = 1
                patterns.append(k)

    if rotate_invariant:
        # 重複を回転で除去
        unique = []
        for p in patterns:
            rots = [np.rot90(p, k) for k in range(4)]
            if not any(np.array_equal(r, u) for r in rots for u in unique):
                unique.append(min(rots, key=lambda x: x.tobytes()))
        return unique
    else:
        return patterns

def extract_hlac_features(image, order=2, rotate_invariant=False, normalize=True, scales=[1.0, 0.75, 0.5]):
    """スケール不変・正規化HLAC特徴抽出"""
    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    kernels = hlac_kernels(order=order, rotate_invariant=rotate_invariant)
    total_features = np.zeros(len(kernels), dtype=np.float32)

    for scale in scales:
        scaled = cv2.resize(image, dsize=None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        _, binary = cv2.threshold(scaled, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        padded = np.pad(binary, pad_width=1, mode='constant')

        for i, k in enumerate(kernels):
            conv = convolve2d(padded, k[::-1, ::-1], mode='valid')
            total_features[i] += np.sum(conv == k.sum())

    if normalize:
        total = np.sum(total_features)
        if total > 0:
            total_features /= total

    return total_features

In [ ]:
# 向きの変化（回転）を無視したい場合は True
image = cv2.imread("sample.bmp")
features = extract_hlac_features(
    image,
    order=2,
    rotate_invariant=True,
    normalize=True,
    scales=[1.0, 0.75, 0.5]
)

print("特徴次元数:", len(features))
print("HLAC特徴量（スケール不変+正規化）:", features)
